# Denoising Diffusion Implict Models

可能让你惊讶，一个 DDPM 框架下训练的模型，不需要任何改动就可以用来做 DDIM 式的推理。换句话说，DDPM 和 DDIM 学习的噪声分布完全相同，目标函数完全相同，训练过程完全相同。唯一的不同是采取了不同的推理方式。但是这一推理方式的转变却极大程度提升了性能。一个比喻是数学家为 Riemann 积分打上反常积分这一补丁，或者把 Lebesgue 积分的领域扩张到 Lebesgue-Stieltjes 积分。我为你讲述。

### 基本逻辑

推荐你读 https://arxiv.org/abs/2010.02502 Denoising Diffusion Implicit Models 这是 DDIM 的原论文。

重新思考 DDPM 的目标函数$$ L = \mathbb{E}_q \left[ -\ln \frac{p_\theta(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right]$$

如果你仅仅观察这个形式，你会认为这个目标函数毫无疑问与 Markovian 链每个单步相关。但是其实我们早在上一章就埋下伏笔，在上一章对于目标函数的简化中，我们最终把这个结果简化到预测噪声与真实噪声之间的距离平方期望。

是否记得$$L_{simple} = \sum_{t=1}^T \mathbb{E}_{\mathbf{x}_0 \sim q(\mathbf{x}_0), \boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})} \left[ \| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \|^2 \right]$$

代入我们的结果$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}$$
得到
$$L_{simple} = \sum_{t=1}^T \mathbb{E}_{\mathbf{x}_0 \sim q(\mathbf{x}_0), \boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})} \left[ \| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\underbrace{\sqrt{\bar{\alpha}_t}\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\boldsymbol{\epsilon}}_{\mathbf{x}_t \text{ 的边缘采样}}, t) \|^2 \right]$$

在这个最终的训练目标里，没有任何一项涉及到 $\mathbf{x}_{t-1}$ 或路径上的其他点。模型只看到了 $\mathbf{x}_0$（数据边缘）和 $\mathbf{x}_t$（由边缘分布 $q(\mathbf{x}_t|\mathbf{x}_0)$ 生成）。所以训练最终只要求我们能从 $q(\mathbf{x}_t|\mathbf{x}_0)$ 里采样，并不要求中间是如何加噪的。

既然 $L$ 只取决于边缘分布 $q(\mathbf{x}_t | \mathbf{x}_0)$，那么我们可以反向设计前向过程。我们可以保留相同的边缘分布（为了复用 DDPM 训练好的模型），但改变联合分布 $q(\mathbf{x}_{1:T} | \mathbf{x}_0)$。

我们定义全新的联合分布$$q_\sigma(\mathbf{x}_{1:T}|\mathbf{x}_0) := q_\sigma(\mathbf{x}_T|\mathbf{x}_0) \prod_{t=2}^T q_\sigma(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0)$$
其中 $q_{\sigma}(\boldsymbol{x}_T|\boldsymbol{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_T}\boldsymbol{x}_0, (1 - \bar{\alpha}_T)\boldsymbol{I})$

注意乘积项里的 $q_\sigma(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0)$。在 DDPM 里，从 $x_t$ 变到 $x_{t-1}$ 是不需要知道 $x_0$ 的。非 Markovian 的本质体现在，每一个中间状态 $x_{t-1}$ 都直接依赖于原始图像 $x_0$。这意味着前向过程的每一步都在根据终点来移动。

为了让这种新的过程能够复用 DDPM 训练好的权重，我们必须保证每一时刻的边缘分布 $q(x_t|x_0)$ 依然是那个高斯分布。定义 $$q_\sigma(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}\left( \underbrace{\sqrt{\bar{\alpha}_{t-1}} \mathbf{x}_0}_{\text{预测的 } \mathbf{x}_0 \text{ 部分}} + \underbrace{\sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \frac{\mathbf{x}_t - \sqrt{\bar{\alpha}_{t}} \mathbf{x}_0}{\sqrt{1 - \bar{\alpha}_{t}}}}_{\text{指向当前的修正}}, \sigma_t^2 \mathbf{I} \right)$$
其中 $\bar{\alpha}_t$ 对应 DDPM 中的累乘系数。

解释一下，这个构造是刻意为之的。实际上是为了保持$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}$$更多的，$\mathbf{x}_0$ 项确定了我们要回归到的目标图像。$\frac{\mathbf{x}_t - \sqrt{\alpha_t} \mathbf{x}_0}{\sqrt{1 - \alpha_t}}$ 实际上就是训练时加入的噪声 $\boldsymbol{\epsilon}$。这一项代表了当前状态中指向噪声的方向。最后 $\sigma_t$ 项是随机性的开关。

注意最后这个随机性的开关。当 $\sigma_t$ 满足特定值时，这个非马尔可夫过程在数学上会塌缩回 DDPM。此时它依然是一个随机游走的过程。当 $\sigma_t = 0$ 时，方差项消失。整个前向（以及逆向）过程变成了完全确定性的。给定一个 $x_t$，其上一步 $x_{t-1}$ 是可以由公式计算出来的。

你可能明白 Implicit Models 这个名字的由来。DDIM 指出，$\sigma_t$ 设置为 0 的确定性带来了巨大的便利，包括对于任意原始噪声数据的确定性推理。

这里有问题。既然我们设计了一套依赖 $x_0$ 的前向过程，那在推理（采样）时根本没有 $x_0$ 该怎么处理？原论文作者定义了一个函数 $f_\theta^{(t)}(\mathbf{x}_t)$ 反向估计 $$f_\theta^{(t)}(\mathbf{x}_t) := (\mathbf{x}_t - \sqrt{1 - \alpha_t} \cdot \boldsymbol{\epsilon}_\theta^{(t)}(\mathbf{x}_t)) / \sqrt{\alpha_t}$$ 
你会发现这个公式本质是 $\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}$ 的变形

于是可以得到 $$p_\theta^{(t)}(\mathbf{x}_{t-1}|\mathbf{x}_t) = q_\sigma(\mathbf{x}_{t-1} | \mathbf{x}_t, f_\theta^{(t)}(\mathbf{x}_t))$$

我们重新证明 DDIM 的目标函数本质与 DDPM 相同。原论文中目标函数是 $$J_\sigma(\epsilon_\theta) := \mathbb{E}_{\mathbf{x}_{0:T} \sim q_\sigma(\mathbf{x}_{0:T} | \mathbf{x}_0)} [\ln q_\sigma(\mathbf{x}_{1:T} | \mathbf{x}_0) - \ln p_\theta(\mathbf{x}_{0:T})]$$
你应该对这个目标函数很熟悉。从形式上看，他和 DDPM 是一模一样的，其实动机推导也差不多，我不赘述。

自然的，我们可以证明对于任何 $\sigma > 0$，上述复杂的 $J_\sigma$ 都可以塌缩为一个简单的加权均方误差之和：$$L_\gamma = \sum_{t=1}^T \gamma_t \mathbb{E}_{\mathbf{x}_0 \sim q(\mathbf{x}_0), \boldsymbol{\epsilon}_t \sim \mathcal{N}(\mathbf{0}, \mathbf{I})} [\|\boldsymbol{\epsilon}_t - \boldsymbol{\epsilon}_\theta^{(t)}(\sqrt{\alpha_t} \mathbf{x}_0 + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_t, t)\|^2] + C$$

但是，我必须为你演示这个证明。因为虽然形式上这个 $L_{\gamma}$ 与我们在 DDPM 中提到的 $L_{simple}$ 高度相似，但是本质很不同，我们定义分布的方式并不一样。这和刚刚说的目标函数不一样，因为目标函数的推导动机都是最大化 $p_{\theta}(x_0)$，这一点 DDIM 和 DDPM 是共通的。

展开$$J_{\sigma} = \mathbb{E}_q [ \underbrace{D_{KL}(q(\mathbf{x}_T | \mathbf{x}_0) \| p(\mathbf{x}_T))}_{L_T} + \sum_{t=2}^T \underbrace{D_{KL}(q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) \| p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t))}_{L_{t-1}} \underbrace{- \ln p_\theta(\mathbf{x}_0 | \mathbf{x}_1)}_{L_0} ]$$
简化 $L_{t-1}$ $$L_{t-1} = \mathbb{E} \left[ \frac{1}{2\sigma_t^2} \| \boldsymbol{\mu}_q - \boldsymbol{\mu}_p \|^2 \right]$$

代入真实后验均值 $$\boldsymbol{\mu}_q = \sqrt{\bar{\alpha}_{t-1}} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \frac{\mathbf{x}_t - \sqrt{\bar{\alpha}_t} \mathbf{x}_0}{\sqrt{1 - \bar{\alpha}_t}}$$
或者我们化简一下 $$\boldsymbol{\mu}_q = \sqrt{\bar{\alpha}_{t-1}} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \boldsymbol{\epsilon}_t$$

模型通过神经网络预测噪声 $\boldsymbol{\epsilon}_\theta^{(t)}$，并估计出原图 $f_\theta^{(t)}(\mathbf{x}_t)$。代入同样的结构：$$\boldsymbol{\mu}_p = \sqrt{\bar{\alpha}_{t-1}} f_\theta^{(t)}(\mathbf{x}_t) + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \boldsymbol{\epsilon}_\theta^{(t)}$$

系列化简得到 $$\boldsymbol{\mu}_q - \boldsymbol{\mu}_p = \left( \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} - \frac{\sqrt{\bar{\alpha}_{t-1}}\sqrt{1 - \bar{\alpha}_t}}{\sqrt{\bar{\alpha}_t}} \right) (\boldsymbol{\epsilon}_t - \boldsymbol{\epsilon}_\theta^{(t)})$$

实际上已经得证$$L_{t-1} = \mathbb{E} \left[ \frac{(\text{关于 } t, \sigma \text{ 系数})^2}{2\sigma_t^2} \| \boldsymbol{\epsilon}_t - \boldsymbol{\epsilon}_\theta^{(t)} \|^2 \right]$$

我们已经基本说完了。总之你可以看到，DDIM 从边缘概率出发反向设计了整个前向加噪过程，去除了 DDPM 不必要的 Markovian 链，解放了多步推理。下面我们来详谈推理是如何进行的。

### 推理

初始时，我们有噪声图像 $\mathbf{x}_{t}$ 和时间步 $t$。$\mathbf{x}_{t}$ 是一个纯噪声张量，从 $\mathcal{N}(0, I)$ 采样得来。$t$ 是当前的时间步索引。

交给网络推理，得到预测噪声 $$\boldsymbol{\epsilon}_\theta = \text{Model}(\mathbf{x}_{t}, t)$$

反向预测原始图像 $$\text{pred\_}\mathbf{x}_0 = \frac{\mathbf{x}_{t} - \sqrt{1 - \bar{\alpha}_{t}} \boldsymbol{\epsilon}_\theta}{\sqrt{\bar{\alpha}_{t}}}$$

得到下一个跳跃的时间步的张量 $$\mathbf{x}_{t-{\Delta t}} = \underbrace{\sqrt{\bar{\alpha}_{t-\Delta t}} \cdot \text{pred\_}\mathbf{x}_0}_{\text{信号部分}} + \underbrace{\sqrt{1 - \bar{\alpha}_{t-\Delta t} - \sigma^2} \cdot \boldsymbol{\epsilon}_\theta}_{\text{方向部分}} + \underbrace{\sigma \boldsymbol{\epsilon}_{rand}}_{\text{随机部分}}$$
其中 $\Delta t$ 是一个不超过 $t$ 的正整数。

所以你可以看到，通过合法的跳步，DDIM 可以大大压缩推理步数，甚至可以决定生成快慢。

下面我想说一说本质。我想我们很早就提到了，Score Matching，NSCN 和 DDIM 本质是同源的，那就是解决一个常微分方程。这一段在 DDIM 的原论文最后被简短地指出了。我们说说。

### DDIM 的本质

我们先给出完整的推理公式，就是上述推理过程的整合$$\mathbf{x}_{t-{\Delta t}} = \underbrace{\sqrt{\bar{\alpha}_{t-{\Delta t}}} \left( \frac{\mathbf{x}_t - \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}_\theta^{(t)}(\mathbf{x}_t)}{\sqrt{\bar{\alpha}_t}} \right)}_{\text{predicted} \mathbf{x}_0} + \underbrace{\sqrt{1 - \bar{\alpha}_{t-{\Delta t}} - \sigma_t^2} \cdot \boldsymbol{\epsilon}_\theta^{(t)}(\mathbf{x}_t)}_{\text{pointing to } \mathbf{x}_t} + \underbrace{\sigma_t \boldsymbol{\epsilon}_t}_{\text{random noise}}$$

便利我们观察，我们做代换。令 $\bar{\mathbf{x}} = \mathbf{x}/\sqrt{\alpha}$，同时令 $\sigma_t$ 为 0 保证完全确定。

$$\frac{\boldsymbol{x}_{t-\Delta t}}{\sqrt{\bar{\alpha}_{t-\Delta t}}} = \frac{\boldsymbol{x}_t}{\sqrt{\bar{\alpha}_t}} + \left( \sqrt{\frac{1 - \bar{\alpha}_{t-\Delta t}}{\bar{\alpha}_{t-\Delta t}}} - \sqrt{\frac{1 - \bar{\alpha}_t}{\bar{\alpha}_t}} \right) \epsilon_{\theta}^{(t)}(\boldsymbol{x}_t)$$

令 $\sigma(t) = \frac{\sqrt{1-\alpha_t}}{\sqrt{\alpha_t}}$。关于这里的 $\sigma(t)$ 和先前的 $\sigma_t$ 长得很像，我不得已为之，因为 DDIM 原论文记号就是这样，请你记住他们不同 $$\bar{\mathbf{x}}_{t-{\Delta t}} = \bar{\mathbf{x}}_t + (\sigma_{t-{\Delta t}} - \sigma_t) \boldsymbol{\epsilon}_\theta$$

你会发现这很像解 ODE 的 Euler Method $$y(t+\Delta t) = y(t) + f(t, y) \Delta t$$

当我们让采样的步数 $T \to \infty$ 时，每一步的变化量 $\Delta \sigma \to 0$，时间步长 $\Delta t \to 0$ 时，离散的欧拉步就变成了连续的 ODE $$\bar{\mathbf{x}}_{t-1} - \bar{\mathbf{x}}_t \implies d\bar{\mathbf{x}}$$ $$\sigma_{t-1} - \sigma_t \implies d\sigma$$现在的结果是 $$d\bar{\mathbf{x}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) d\sigma$$ 或者我们换一种等价写法$$\mathrm{d}\bar{\mathbf{x}}(t) = \epsilon_\theta^{(t)} \left( \frac{\bar{\mathbf{x}}(t)}{\sqrt{\sigma^2 + 1}} \right) \mathrm{d}\sigma(t)$$

我们指出，图像的演化（$\mathrm{d}\bar{\mathbf{x}}$）是沿着神经网络预测的噪声方向（$\epsilon_\theta$）进行的，而演化的推进动力是信噪比的变化（$\mathrm{d}\sigma$）。在这个 ODE 中，我们的起点是 $t$ 时刻的纯噪声，终点是 $0$ 时刻的干净图像。

关于这个 ODE 本体，我为你展示。实际上这个 ODE 非常著名，被称为 Probability Flow ODE $$\frac{d\mathbf{x}}{dt} = \underbrace{ \frac{1}{2} \frac{d \ln \alpha(t)}{dt} \mathbf{x} }_{\text{漂移项 (信号衰减)}} - \underbrace{ \frac{1}{2} \frac{d \sigma^2(t)}{dt} \frac{1}{\sqrt{1-\alpha(t)}} \boldsymbol{\epsilon}_\theta(\mathbf{x}, t) }_{\text{得分项 (去噪推力)}} $$

回顾我们的 Score Matching。实际上在数学上，噪声预测器 $\boldsymbol{\epsilon}_\theta$ 与数据分布的得分函数（Score Function） $\nabla_\mathbf{x} \log p_t(\mathbf{x})$ 满足以下等式 $$\nabla_\mathbf{x} \log p_t(\mathbf{x}) = -\frac{\boldsymbol{\epsilon}_\theta(\mathbf{x}, t)}{\sqrt{1-\alpha_t}}$$

代入后可以得到 $$d\mathbf{x} = \left[ \mathbf{f}(\mathbf{x}, t) - \frac{1}{2} g(t)^2 \nabla_\mathbf{x} \log p_t(\mathbf{x}) \right] dt$$

最后的 ODE 我们说得太快了。在下一章我们会详谈这个 SM 与 DDIM 的统一。

# 总结

本章我们介绍了 DDPM 的升级版本 DDIM。下一章中我为你串联 SM 与 DDIM 的本质，你会意识到他们在解决同一个问题。我们进入下一章。